# GeoMx Analysis and Data Visualization

**Author: Fabien Bertillot**  

---
## 1. Objective

Analysis of GeoMx spatial transcriptomic data from colorectal tumor samples across 8 patients, with focus on tissue architecture, molecular clustering, and pathway enrichment.

## 2. Dependencies

- Libraries: `scanpy`, `pandas`, `community`, `matplotlib`, `seaborn`, `networkx`, `seaborn`, `gseapy`, `pydeseq2`
- Files required:  
  - `data.h5ad` - Dataset format must be an AnnData
  - `gene_sets.json` - A nested dictionary of gene set signatures
---

## 3. Notebook Roadmap

1. **Load Dependencies**  
2. **Load Dataset**  
   - Read `data.h5ad`  
   - Read `gene_sets.json`  
   - Filter some samples and define some colormaps
3. **Scanpy Preprocessing Parameters**  
4. **Harmony integration for batch correction**  
    - Preprocessing and Batch correction
    - Leiden clustering
    - UMAP plots
5. **Preprocessing and clustering (without batch correction)**  
    - Preprocessing and Leiden clustering
    - UMAP plots
6. **Spatial Clustering**  
    - Cluster GeoMx ROIs into spatial regions with Louvain clustering
    - Plot molecular clusters and architectures across spatial clusters
7. **Plot Number of Regions Across Patients** 
8. **Stacked Bar Plots (patients, architectures, molecular clusters)** 
9. **Projection of GeoMX ROIs Colored by Tissue Architecture and Molecular Clusters** 
10. **Compute Differential Gene Expression with pydeseq2**
11. **Gene Set Enrichment Analysis (GSEA)**
    - Compute GSEA for given signatures
    - Plots
12. **Box Plots (Gene expression across patients, architectures)**

---

## 1) Load Dependencies

In [ ]:
from pathlib import Path

import pandas as pd
import anndata as ad
import numpy as np 
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import networkx as nx
import community as community_louvain
import seaborn as sns
import json
import os
# from adjustText import adjust_text
from geomxlib import *
import matplotlib
import gseapy as gp

In [ ]:
from watermark import watermark
%reload_ext watermark
%watermark --iversions
# %watermark -iv

## 2) Load Dataset

In [ ]:
DATA_DIR = Path(r"../data/geomx")
FIGURE_DIR = Path(r"../outputs/geomx/figures")
# Extra DESeq2 design factors; the paper used none. Set to ['patient_id'] to adjust for patient.
DGE_COVARIATES = []

In [ ]:
# papermill passes parameters as strings
DATA_DIR = Path(DATA_DIR)
FIGURE_DIR = Path(FIGURE_DIR)

# -- Load anndata and get sets

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
EXPRS_PATH = DATA_DIR / 'geomx_crc_dataset.h5ad'
GENE_SETS_PATH = DATA_DIR / 'gene_sets.json'
DGE_PATH = DATA_DIR / 'raw_dges_architectures.csv'

adata = sc.read(EXPRS_PATH)

with open(GENE_SETS_PATH, 'r') as file:
    gene_sets = json.load(file)
    
    
# -- Filter some samples 
adata_raw = adata[adata.obs['segment'] == 'Segment 1'] # Filter stroma
adata = adata_raw.copy()

# -- Define some colormaps 

# Architectures
architecture_order = [
    'solid',
    'glandular dense',
    'glandular',
    'glandular infiltrated',
    'mod. fragmented',
    'fragmented',
]
architecture_palette = ['#009ADE','#00CD6C', '#FFC61E', '#FED976','#F28522','#FF1F5B']
architecture_colormap = dict(zip(architecture_order, architecture_palette))

# Patients
patient_order = [
    'P1',
    'P2',
    'P3',
    'P4',
    'P5',
    'P6',
    'P7',
    'P8',
]
patient_palette = []
patient_colormap = dict(zip(patient_order, sns.color_palette('Set2')))

In [ ]:
adata

## 3) Scanpy Preprocessing Parameters

In [ ]:
TOP_GENES = 2000
N_NEIGHBORS = 10
N_PCS = 30
RES = 2
KEY_LEIDEN = 'leiden_2'

## 4) Harmony Integration

In [ ]:
# -- Batch correction with harmony

adata_raw_hry = adata_raw.copy()

sc.pp.log1p(adata_raw_hry, layer='processed')
adata_raw_hry.X = adata_raw_hry.layers['processed']    

sc.pp.highly_variable_genes(adata_raw_hry, n_top_genes=TOP_GENES, batch_key="patient_id")

sc.pp.pca(adata_raw_hry,  svd_solver='arpack')
sc.external.pp.harmony_integrate(adata_raw_hry , key='patient_id')

sc.pp.neighbors(adata_raw_hry, use_rep='X_pca_harmony', n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata_raw_hry)

In [ ]:
# -- Leiden clustering

sc.tl.leiden(adata_raw_hry, resolution=RES, key_added=KEY_LEIDEN)

In [ ]:
# -- UMAP plots

sns.set(font_scale = 2)
sns.set_style('white', rc={
'xtick.bottom': True,
'ytick.left': True,
})
plt.rcParams["figure.figsize"] = (5,5)
sc.pl.umap(adata_raw_hry, color=[KEY_LEIDEN], show=False,size=70,
        #    palette={str(i):c for i,c in enumerate(sns.color_palette('tab20', 20))}
           )
ax = plt.gca()

# Remove the top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title('')

plt.show()
sc.pl.umap(
    adata_raw_hry,
    color=['patients_ordered'],
    size=70,
    show=False,
    alpha=0.9,
    palette=patient_colormap,
)
ax = plt.gca()

# Remove the top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title('')

sc.pl.umap(adata_raw_hry, color=['architectures'], palette=architecture_colormap, size=300)
sc.pl.umap(adata_raw_hry, color=['ANXA1'], palette=architecture_colormap, size=300)

## 5) Preprocessing and Clustering

### 5.a) Preprocessing and clustering

In [ ]:
# -- Leiden preprocessing

adata = scanpy_preprocessing(adata, 
                             layer='processed',
                             n_neighbors=N_NEIGHBORS,
                             n_pcs=N_PCS)



In [ ]:
# -- Leiden clustering

sc.tl.leiden(adata, resolution=RES, key_added=KEY_LEIDEN)

### 5.b) UMAP projection

In [ ]:
sns.set(font_scale = 2)
sns.set_style('white', rc={
'xtick.bottom': True,
'ytick.left': True,
})
plt.rcParams["figure.figsize"] = (5,5)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.pl.umap(adata, color=['patients_ordered'], size=70, show=False, alpha=0.9, palette=patient_colormap)
ax = plt.gca()

# Remove the top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title('')

# OUT_DIR = str(FIGURE_DIR)
# out_path = os.path.join(OUT_DIR, f'tumor_UMAP_patients.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


### 5.c) Project molecular clusters on UMAP

In [ ]:
sns.set(font_scale = 2)
sns.set_style('white', rc={
'xtick.bottom': True,
'ytick.left': True,
})
plt.rcParams["figure.figsize"] = (5,5)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

leiden_colormap = {str(i):c for i,c in enumerate(sns.color_palette('tab20', 20))}

sc.pl.umap(adata, color=[KEY_LEIDEN], show=False,size=70,
           palette=leiden_colormap
           )
ax = plt.gca()

# Remove the top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title('')

# plt.show()

# OUT_DIR = str(FIGURE_DIR)
# out_path = os.path.join(OUT_DIR, f'tumor_UMAP_molecular_clusters.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')

## 6) Spatial Clustering

### 6.a) Louvain clustering

In [ ]:
def louvain_clustering(adata, patient_id_col, x_col, y_col):
    # Get unique patient IDs
    patient_ids = adata.obs[patient_id_col].unique()
    
    # Prepare a Series to store combined cluster labels
    combined_clusters = pd.Series(index=adata.obs.index, dtype='object')
    
    for patient_id in patient_ids:
        # Subset the data for the current patient
        subset = adata[adata.obs[patient_id_col] == patient_id]
        
        # Extract the spatial coordinates
        coords = subset.obs[[x_col, y_col]].values
        
        # Create a nearest neighbors graph
        neighbors = NearestNeighbors(n_neighbors=10).fit(coords)
        graph = neighbors.kneighbors_graph(coords)
        
        # Convert the neighbors graph to a NetworkX graph
        # G = nx.from_scipy_sparse_matrix(graph)
        graph_coo = graph.tocoo()
        G = nx.Graph()
        G.add_edges_from(
            (row, col, {"weight": weight})
            for row, col, weight in zip(graph_coo.row, graph_coo.col, graph_coo.data)
            if row != col
)
        
        # Run Louvain clustering
        partition = community_louvain.best_partition(G)
        
        # Combine patient ID with cluster value
        combined_cluster_labels = {idx: f"{patient_id}_{cluster}" for idx, cluster in partition.items()}
        
        # Store the combined cluster labels
        clusters_subset = pd.Series(combined_cluster_labels, name='combined_cluster')
        clusters_subset.index = subset.obs.index
        combined_clusters.update(clusters_subset)
    
    # Add the combined clusters back to the original AnnData object
    adata.obs['combined_cluster'] = combined_clusters
    
    return adata

In [ ]:
adata = louvain_clustering(adata, 'patient_id', 'X', 'Y')

### 6.b) Plot leiden cluster distribution across spatial clusters

In [ ]:
selected_patients = [
                    #  'P1',
                     'P2', 
                     'P3',
                    #  'P4', 
                     'P5', 
                     'P6', 
                    #  'P7', 
                     'P8'
                     ]

adata_filt = adata[adata.obs['patient_id'].isin(selected_patients)]
ax = barplot(adata_filt, 
             x='combined_cluster',
             labels='leiden_2', 
            #  bar_order = patient_order,
             colormap = leiden_colormap,
             norm=False,
             figsize = (12,3),
             kind='bar',
             rotation=90
            )


plt.xlabel('Spatial clusters')
plt.ylabel('count')

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'molecular_clusters_across_spatial_clusters.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


### 6.c) Plot architecture  distribution across spatial clusters

In [ ]:
# -- Plot architecture  distribution across spatial clusters
selected_patients = [
                    #  'P1',
                     'P2', 
                     'P3',
                    #  'P4', 
                     'P5', 
                     'P6', 
                    #  'P7', 
                     'P8'
                     ]

adata_filt = adata[adata.obs['patient_id'].isin(selected_patients)]

ax = barplot(adata_filt, 
             x='combined_cluster',
             labels='architectures', 
            #  bar_order = patient_order,
             colormap = architecture_colormap,
             norm=False,
             figsize = (12,3),
             kind='bar',
             rotation=90
            )


plt.xlabel('')
plt.ylabel('count')

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'architectures_across_spatial_clusters.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')

## 7) Plot Number of Regions Across Patients

In [ ]:
patient_count = adata.obs['patient_id'].value_counts()

# Plotting
plt.figure(figsize=(5, 5))
patient_count = patient_count.reindex(patient_order)

patient_count.plot(kind='barh')
plt.xlabel('# regions')
sns.despine()
plt.xlim([0,50])
# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'stroma_nb_rois_per_patient.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


In [ ]:
ct_im_colormap = {'CT':'#009392', 'IM':'#CF597E'}

ax = barplot(adata_raw, 
             x='patients_ordered',
             labels='region', 
            #  bar_order = patient_order,
            norm=False,
             colormap = ct_im_colormap,
             figsize = (4,4),
             kind='barh'
            )

plt.xlabel('# regions')
plt.xlim([0,50])

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'stroma_nb_rois_per_patient.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


## 8) Stacked Bar Plots (patients, architectures, molecular clusters) 

In [ ]:
# -- Leiden cluster distribution across patients

leiden_colormap = {str(i):c for i,c in enumerate(sns.color_palette('tab20', 20))}

ax = barplot(adata, 
             x='patients_ordered',
             labels='leiden_2', 
            #  bar_order = patient_order,
            norm=False,
             colormap = leiden_colormap,
             figsize = (4,3),
             kind='barh'
            )


plt.xlabel('%')
plt.ylabel('')

# OUT_DIR = str(FIGURE_DIR)
# out_path = os.path.join(OUT_DIR, f'distribution_clusters_per_patient.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')

# plt.xticks(rotation=90, ha='center')  

In [ ]:
# Distribution of architectures across molecular (leiden) clusters
x = KEY_LEIDEN
ax = barplot(adata, 
             x=x,
             labels='architectures', 
             bar_order = architecture_order,
             colormap = architecture_colormap,
             figsize = (4,6),
             norm=False,
             kind='barh'
            )

plt.xlabel('%')
plt.ylabel('Molecular clusters')

# OUT_DIR = str(FIGURE_DIR)
# out_path = os.path.join(OUT_DIR, f'distribution_architecture_per_molecular_cluster.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')

In [ ]:
# -- Architecture distribution across patients

x = 'patient_id'
ax = barplot(adata, 
             x=x,
             x_order = patient_order,
             labels='architectures', 
             bar_order = architecture_order,
             colormap = architecture_colormap,
             figsize = (3,4),
             kind='barh'
            )

plt.xlabel('')
plt.ylabel('')

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'distribution_architecture_per_patient.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


## 9) Projection of GeoMX ROIs Colored by Tissue Architecture and Molecular Clusters

In [ ]:
# -- Plot leiden cluster spatial distribution of a given patient

PATIENT_ID = 'P6'
print(f'projection on {PATIENT_ID}')


project_data_into_tissue(adata, 
                        patient_id=PATIENT_ID, 
                        color=KEY_LEIDEN, 
                        colormap=leiden_colormap,
                        fontsize=18)

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'molecular_cluster_projection_{PATIENT_ID}.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')

In [ ]:
# -- Plot architecture spatial distribution of a given patient

PATIENT_ID = 'P6'
print(f'projection on {PATIENT_ID}')

plt.figure(2)

project_data_into_tissue(adata, 
                        patient_id=PATIENT_ID, 
                        color='architectures', 
                        colormap=architecture_colormap,
                        fontsize=18)

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'architecture_projection_{PATIENT_ID}.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


## 10) Compute DGE with pydeseq2

In [ ]:
# -- Compute DGE with pydeseq2 (can take up to 10min)

condition = 'architectures'
OUT_PATH = FIGURE_DIR / 'raw_dges_architectures_simplified.csv'

counts_df = pd.DataFrame(data=adata_raw.X, 
                         index=adata_raw.obs['sample_id'],
                         columns=adata_raw.var_names)

metadata = pd.DataFrame({
                         condition: adata_raw.obs[condition].values,
                         'patient_id': adata_raw.obs['patient_id'].values
                        },  index=adata_raw.obs['sample_id'])


conditions = adata_raw.obs[condition].unique()

dfs_dge = []

for cond in conditions:
    print(cond)
    df_cond = compute_dge_from_pydeseq2(counts_df, metadata, condition, cond, covariates=DGE_COVARIATES)
    dfs_dge.append(df_cond)
    
    
dfs_dge = pd.concat(dfs_dge).reset_index(drop=True)

dfs_dge.to_csv(OUT_PATH, index=False)

## 11) GSEA Analysis

In [ ]:
# -- Load DGE file

df_dge = pd.read_csv(DGE_PATH)
df_dge.head(10)

In [ ]:
# -- Compute GSEA

stratify='architectures'
gene_set = gene_sets['STEM_signatures']

conditions=architecture_order
# conditions=['glandular','solid','fragmented']

df_gsea = []
for condition in conditions:
    df_rnk = df_dge[df_dge['group']==condition]
    # df_rnk = df_rnk[df_rnk['lfc']>0.1]
    # df_rnk = df_rnk[df_rnk['pval_adj']<0.05]
    df_rnk = df_rnk.sort_values(by='pval_adj')

    df_rnk = df_rnk[['Gene','lfc']].reset_index(drop=True)

    pre_res = gp.prerank(rnk=df_rnk, # or rnk = rnk,
                        gene_sets=gene_set,
                        threads=4,
                        min_size=5,
                        max_size=1000,
                        permutation_num=1000, # reduce number to speed up testing
                        outdir=None, # don't write to disk
                        seed=6,
                        verbose=True, # see what's going on behind the scenes
                        )

    pre_res.res2d['group'] = [condition]*len(pre_res.res2d)
    df_gsea.append(pre_res.res2d)

df_gsea = pd.concat(df_gsea).reset_index(drop=True)

In [ ]:
# -- Plot GSEA

index_order = ['Crypt base', 'LGR5 ISC', 'Fetal (human)', 'RSC', 'revCSC', 'EpiHR', 'coreHRC']
# index_order = [
#     'Wound response', 'Mesenchymal', 'Matrix remodeling',
#     'TGFB activation', 'EMT activation', 'EMT Hallmark',
#     'pEMT', 'EMT-II (MP13)',
# ]

FIG_SIZE = (8, 6)

def map_pvalues_to_sizes(pval):
    if pval < 0.05:
        return 500
    elif pval < 0.1:
        return 300
    elif pval < 0.25:
        return 150
    # elif pval < 0.05:
    #     return 100
    elif pval > 0.25:
        return 0
    else:
        return 0

df_pivot_es = df_gsea.pivot(index="Term", columns="group", values="NES").reset_index()
df_pivot_pval = df_gsea.pivot(index="Term", columns="group", values="FDR q-val").reset_index()

df_pivot_es = df_pivot_es[df_pivot_es['Term'].isin(index_order)]
df_pivot_pval = df_pivot_pval[df_pivot_pval['Term'].isin(index_order)]
# df_pivot_es = df_pivot_es.reindex(index_order)
# df_pivot_pval = df_pivot_pval.reindex(index_order)

df_pivot_es.set_index('Term', inplace=True)
df_pivot_pval.set_index('Term', inplace=True)
df_pivot_es = df_pivot_es.reindex(index_order)
df_pivot_pval = df_pivot_pval.reindex(index_order)


df_pivot_es = df_pivot_es[architecture_order]
df_pivot_pval = df_pivot_pval[architecture_order]

df_pivot_pval_sizes = df_pivot_pval.applymap(map_pvalues_to_sizes)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sns.set(font_scale = 2)
sns.set_style('white', rc={
    'xtick.bottom': True,
    'ytick.left': True,
})
fig, ax = plt.subplots(figsize=FIG_SIZE)


terms = df_pivot_es.index
categories = df_pivot_es.columns

for i, term in enumerate(terms):
# for i, term in zip([0.3,0.7],terms):
    y = np.repeat(i, len(categories))  # Y position for all categories
    x = np.arange(len(categories))     # X position for each category
    color_values = df_pivot_es.loc[term]  # Color from the expression score DataFrame
    dot_sizes = df_pivot_pval_sizes.loc[term]  # Sizes from the custom function

    scatter = ax.scatter(x, y, s=dot_sizes, c=color_values, cmap='PiYG', edgecolor='black', alpha=1)

# Adding labels and formatting
ax.set_yticks(np.arange(len(terms)))
# ax.set_yticks([0.3,0.7])

ax.set_yticklabels(terms)
# ax.set_ylim([-0.5,1.5])
ax.set_xlim([-0.5,5.5])

ax.set_xticks(np.arange(len(categories)))
ax.set_xticklabels(categories, rotation=90, ha='center')
ax.invert_yaxis()  # Invert axis to have terms on y-axis in descending order

# Color bar for expression scores
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('NES')

# Legend
legend_elements = [
    plt.Line2D(
        [0], [0], marker='o', color='w',
        label='FDR < 5%', markerfacecolor='gray', markersize=25,
    ),
    plt.Line2D(
        [0], [0], marker='o', color='w',
        label='FDR < 10%', markerfacecolor='gray', markersize=18,
    ),
    plt.Line2D(
        [0], [0], marker='o', color='w',
        label='FDR < 25%', markerfacecolor='gray', markersize=12,
    ),
]
ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(2, 1), frameon=False)


# plt.title('Dot Plot of Terms vs Categories')
# plt.grid(True, which='both', ls='', lw=0.5)
# plt.show()
sns.despine()
OUT_DIR = str(FIGURE_DIR)
out_path = os.path.join(OUT_DIR, f'GSEA_STEM_signatures.pdf')
plt.savefig(out_path, format='pdf', bbox_inches='tight')



## 12) Box Plots (Gene expression across patients, architectures) 

### 12.a) Gene expression across patients

In [ ]:
# -- Plot gene expression across patients

obs_df = adata_raw.obs
var_df = adata_raw[:].to_df()
var_df.iloc[:, :] = adata.layers['processed']
plot_df = pd.concat([obs_df, var_df], axis=1)
feat= 'H2BC3'
plot_df['area'] = plot_df['area']*0.28
# plot_df = plot_df[plot_df['patient_id'] =='P3']

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

boxplot(df=plot_df, 
        x='patient_id',
        y=feat, 
        # order = list(architecture_colormap.keys()),
        xticks_rotation = 90, 
        figsize=(3,3),
        font_scale=1.5,
        alpha = 0.5,
        s=5,
        notch = False,
        showfliers=False,
        stripplot=True,
        swarm_color=list(patient_colormap.values())
)

plt.ylabel('Expression level')

### 12.b) Gene expression across architectures

In [ ]:
# -- Plot gene expression across architectures

obs_df = adata_raw.obs
var_df = adata_raw[:].to_df()
var_df.iloc[:, :] = adata.layers['processed']
plot_df = pd.concat([obs_df, var_df], axis=1)
feat= 'CDX2'
# plot_df['area'] = plot_df['area']*0.28
# plot_df = plot_df[plot_df['patient_id'] =='P3']

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


boxplot(plot_df, 
        x='architectures',
        y=feat, 
        order = list(architecture_colormap.keys()),
        font_scale=1.5,
        xticks_rotation = 90, 
        figsize=(3,3),
        alpha = 0.5,
        s=5,
        notch = False,
        showfliers=False,
        stripplot=True,
        swarm_color=list(architecture_colormap.values())
)

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'{feat}_across_architectures.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')
plt.ylabel('Expression level')

### 12.c) Multiple genes expression across patients

In [ ]:
# -- Plot multiple genes across architectures

obs_df = adata_raw.obs
var_df = adata_raw[:].to_df()
var_df.iloc[:, :] = adata.layers['processed']
plot_df = pd.concat([obs_df, var_df], axis=1)

melted_df = pd.melt(plot_df, id_vars=['architectures'], 
                    value_vars=['LGR5','OLFM4', 'CDX2', 'PROX1'] , 
                    var_name='marker', value_name='value')




matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

plt.rcParams["figure.figsize"] = (10,4)
sns.set(font_scale = 2)
sns.set_style('white', rc={
'xtick.bottom': True,
'ytick.left': True,
    })

g = sns.stripplot(data=melted_df, 
                x='marker',
                y='value', 
                hue="architectures",
                hue_order=architecture_order,
                palette=architecture_palette,
                alpha=0.8,
                s=3,
                jitter=True,
                dodge=True
                )

g = sns.boxplot(data=melted_df, 
                x="marker", 
                y="value",
                hue="architectures",
                hue_order=architecture_order,
                color='w',
                boxprops=dict(linewidth=1, edgecolor='k',facecolor='w'),  # Set edgecolor to black
                medianprops=dict(color='black', linewidth=1),
                )

# for median in g.lines[:]:
#     median.set_color('black')  # Change the color of the median line
#     median.set_linewidth(1)  # Change
    


handles, labels = plt.gca().get_legend_handles_labels()
unique_labels = list(set(labels))
handles = [handles[labels.index(label)] for label in unique_labels]


plt.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, title='')
plt.ylabel('Expression level')

g.tick_params(bottom=True)

plt.xticks(rotation=90)
plt.xlabel('')
sns.despine()

# OUT_DIR = str(FIGURE_DIR / 'sup')
# out_path = os.path.join(OUT_DIR, f'shapes_across_architectures.pdf')
# plt.savefig(out_path, format='pdf', bbox_inches='tight')


In [ ]:
adata_raw.obs.columns